# Trimmed ResNet-18 (4.71M params) distilled from a WideResNet-28-10 teacher

Deep Learning Project 1 (Fall 2026): a modified ResNet under 5M parameters for the CIFAR-10 Kaggle competition, trained from scratch on the provided 50,000 images only.

**Final pipeline (best score)**
1. Train a WideResNet-28-10 teacher from scratch (`TEACHER_wrn28_10`, 36.5M params, never submitted).
2. Distill it into our Trimmed ResNet-18 (`DISTILL_wrn`, 4,707,786 params): 300 epochs, KL loss (T = 4, alpha = 0.9) with teacher and student seeing the same augmented batch.
3. Retrain step 2 on all 50,000 images for the final submission (`FINAL_DISTILL_wrn_full50k`).

Student recipe: TrivialAugment + Cutout 16x16 + RandomGrayscale(0.05), SGD with Nesterov momentum, batch 256, peak LR 0.2 with 5-epoch warmup then cosine decay, weight decay 5e-4 (not on BN or bias), EMA of weights (0.999), multi-crop test-time augmentation (mirror plus two 1-pixel shifts).

**Results** (validation = fixed stratified 5,000-image split, seed 42; Kaggle = public leaderboard, about 50% of the 2,000 test images)

| Run | Params | Validation (EMA + TTA) | Kaggle public |
|---|---|---|---|
| `E2_sgd200` (flip TTA) | 4.71M | 96.10% | 0.873 |
| `E3_sam100` (flip TTA) | 4.71M | not re-scored | 0.865 |
| `E2_sgd200` (multi-crop TTA) | 4.71M | 96.54% | 0.878 |
| `TEACHER_wrn28_10` (not submitted) | 36.48M | 97.34% | n/a |
| **`DISTILL_wrn`** (multi-crop TTA) | **4.71M** | **97.02%** | **0.883** |
| `FINAL_DISTILL_wrn_full50k` | 4.71M | n/a (validation images used for training) | see Kaggle |

**How to reproduce in Colab**
1. `Runtime > Change runtime type > A100 GPU`.
2. Put `data-DL.zip` in `MyDrive/DL-ResNet/` on Google Drive.
3. Set `EXPERIMENT` in the config cell and `Runtime > Run all`, in this order: `TEACHER_wrn28_10` (about 2 to 3 h), `DISTILL_wrn` (about 2 h), `FINAL_DISTILL_wrn_full50k` (about 2.5 h). Checkpoints go to Drive, so a disconnected run resumes where it stopped.

The other presets (`E0_baseline`, `E1_full100`, `E2_sgd200`, `E3_sam100`, `E4_sgd400`, `DISTILL`, `FINAL_*`) reproduce the ablations.

**References:** He et al. 2016 (ResNet); Zagoruyko and Komodakis 2016 (WideResNet); Hinton et al. 2015 and Beyer et al. 2022 (knowledge distillation); Muller and Hutter 2021 (TrivialAugment); DeVries and Taylor 2017 (Cutout); Foret et al. 2021 (SAM); Goyal et al. 2017 (zero-init BN); Jordan 2024 (airbench multi-crop TTA); Recht et al. 2018 and Miller et al. 2021 (distribution shift).

## 1. Configuration
This is the only cell you normally edit.
- `EXPERIMENT` picks a preset from the table above; `SEED` gives a second independent run.
- `QUICK_CHECK = True` runs a 2-epoch smoke test on a small subset (a couple of minutes) to confirm everything works before a long run.
- `OVERRIDES` changes any hyperparameter for one run, for example `{"optimizer": "sam", "epochs": 100}` if SAM wins on Nov 3.
- `TEACHERS` lists the models used by the distillation presets. By default it is the WideResNet-28-10 teacher from step 2. You can add `{"name": "e2", "kind": "ours", "path": "runs/E2_sgd200_seed0/last.pth"}` or Vraj's TorchScript file to make an ensemble teacher.
- The `DL_*` environment variables at the bottom let the same code run as a plain script (for example on NYU HPC) without editing the cell.

In [ ]:
import json
import os

EXPERIMENT = "DISTILL_wrn"
SEED = 0
QUICK_CHECK = False
OVERRIDES = {}

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/DL-ResNet"
DATA_ZIP_NAME = "data-DL.zip"
DATA_DIR_OVERRIDE = ""
RESUME = True
CHECKPOINT_EVERY_EPOCHS = 5
USE_COMPILE = False
AUTO_DOWNLOAD_SUBMISSION = True

TEACHERS = [
    {"name": "wrn28_10", "kind": "ours", "path": "runs/TEACHER_wrn28_10_seed0/last.pth"},
]
KD_TEMPERATURE = 4.0
KD_ALPHA = 0.9

BASE_HPARAMS = {
    "architecture": "resnet18t",
    "role": "student",
    "stage_channels": [64, 128, 256, 224],
    "blocks_per_stage": [2, 2, 2, 2],
    "batch_size": 256,
    "peak_lr": 0.2,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "warmup_epochs": 5,
    "ema_decay": 0.999,
    "sam_rho": 0.05,
    "cutout_size": 16,
    "grayscale_p": 0.05,
    "val_per_class": 500,
    "split_seed": 42,
    "train_subset": None,
    "val_subset": None,
}

PRESETS = {
    "E0_baseline": {"epochs": 100, "augmentation": "basic", "optimizer": "sgd", "train_on_full": False, "distill": False},
    "E1_full100": {"epochs": 100, "augmentation": "full", "optimizer": "sgd", "train_on_full": False, "distill": False},
    "E2_sgd200": {"epochs": 200, "augmentation": "full", "optimizer": "sgd", "train_on_full": False, "distill": False},
    "E3_sam100": {"epochs": 100, "augmentation": "full", "optimizer": "sam", "train_on_full": False, "distill": False},
    "DISTILL": {"epochs": 200, "augmentation": "full", "optimizer": "sgd", "train_on_full": False, "distill": True},
    "FINAL_full50k": {"epochs": 200, "augmentation": "full", "optimizer": "sgd", "train_on_full": True, "distill": False},
    "FINAL_DISTILL_full50k": {"epochs": 200, "augmentation": "full", "optimizer": "sgd", "train_on_full": True, "distill": True},
    "TEACHER_wrn28_10": {"architecture": "wrn28_10", "role": "teacher", "epochs": 200, "batch_size": 128, "peak_lr": 0.1,
                         "augmentation": "full", "optimizer": "sgd", "train_on_full": False, "distill": False},
    "E4_sgd400": {"epochs": 400, "augmentation": "full", "optimizer": "sgd", "train_on_full": False, "distill": False},
    "DISTILL_wrn": {"epochs": 300, "augmentation": "full", "optimizer": "sgd", "train_on_full": False, "distill": True},
    "FINAL_DISTILL_wrn_full50k": {"epochs": 300, "augmentation": "full", "optimizer": "sgd", "train_on_full": True, "distill": True},
}
QUICK_CHECK_HPARAMS = {"epochs": 2, "warmup_epochs": 1, "batch_size": 128, "train_subset": 1024, "val_subset": 500}

EXPERIMENT = os.environ.get("DL_EXPERIMENT", EXPERIMENT)
SEED = int(os.environ.get("DL_SEED", SEED))
QUICK_CHECK = os.environ.get("DL_QUICK_CHECK", "1" if QUICK_CHECK else "0") == "1"
OVERRIDES = json.loads(os.environ["DL_OVERRIDES"]) if "DL_OVERRIDES" in os.environ else OVERRIDES
TEACHERS = json.loads(os.environ["DL_TEACHERS"]) if "DL_TEACHERS" in os.environ else TEACHERS
DATA_DIR_OVERRIDE = os.environ.get("DL_DATA_DIR", DATA_DIR_OVERRIDE)

if EXPERIMENT not in PRESETS:
    raise ValueError(f"Unknown EXPERIMENT '{EXPERIMENT}'. Choose one of: {list(PRESETS)}")

CONFIG = {**BASE_HPARAMS, **PRESETS[EXPERIMENT], **(QUICK_CHECK_HPARAMS if QUICK_CHECK else {}), **OVERRIDES}
CONFIG["experiment"] = EXPERIMENT
CONFIG["seed"] = SEED

RUN_NAME = f"{EXPERIMENT}_seed{SEED}" + ("_quick" if QUICK_CHECK else "")
print(f"Run: {RUN_NAME}")
print(json.dumps(CONFIG, indent=2))

## 2. Install anything that is missing
Colab and Kaggle already ship PyTorch, so this cell usually just confirms. It never reinstalls PyTorch when a working copy is present, because that can break the CUDA build.

In [ ]:
import importlib.util
import subprocess
import sys


def is_installed(module_name):
    return importlib.util.find_spec(module_name) is not None


def pip_install(*packages):
    print("Installing:", " ".join(packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


if not (is_installed("torch") and is_installed("torchvision")):
    pip_install("torch", "torchvision")

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "torchsummary": "torchsummary",
}
missing_packages = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if not is_installed(module_name)]
if missing_packages:
    pip_install(*missing_packages)

import torchvision.transforms as T

if not hasattr(T, "TrivialAugmentWide"):
    pip_install("--upgrade", "torch", "torchvision")
    raise RuntimeError("torchvision was too old and has been upgraded. Use Runtime > Restart session, then Run all again.")

print("All dependencies are available.")

## 3. Imports, GPU check, and precision settings

In [ ]:
import copy
import math
import random
import time
import warnings
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from PIL import Image, ImageEnhance
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IN_COLAB = running_in_colab()
IN_KAGGLE = Path("/kaggle/input").exists()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cpu" and not QUICK_CHECK:
    raise RuntimeError(
        "No CUDA GPU found. In Colab use Runtime > Change runtime type > A100 GPU (TPUs are not supported here), "
        "then Run all again."
    )

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS = min(8, max(2, CPU_COUNT - 1))
USE_AMP = DEVICE.type == "cuda"
SUPPORTS_BF16 = USE_AMP and torch.cuda.get_device_capability()[0] >= 8
AMP_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_memory_gb:.0f} GB), mixed precision: {str(AMP_DTYPE).replace('torch.', '')}")
    if "T4" in gpu_name:
        print("Heads up: a T4 with few CPU cores will be several times slower. A100 is the recommended runtime.")
else:
    print("Running on CPU because QUICK_CHECK is on. This is only a smoke test.")

print(f"torch {torch.__version__}, torchvision {torchvision.__version__}, CPU cores: {CPU_COUNT}, data workers: {NUM_WORKERS}")
if CPU_COUNT < 4 and DEVICE.type == "cuda":
    print("Only a few CPU cores: augmentation may become the bottleneck. A100/H100 runtimes come with more cores.")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

## 4. Storage: Google Drive for checkpoints and results

In [ ]:
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
elif IN_COLAB:
    PROJECT_DIR = Path("/content/DL-ResNet")
elif IN_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working/DL-ResNet")
else:
    PROJECT_DIR = Path.cwd() / "DL-ResNet"

LOCAL_WORK_DIR = Path("/content") if IN_COLAB else Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
CACHE_DIR = PROJECT_DIR / "cache"
for directory in (PROJECT_DIR, RUN_DIR, CACHE_DIR, PROJECT_DIR / "teachers"):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project folder: {PROJECT_DIR}")
print(f"This run saves to: {RUN_DIR}")

## 5. Load the competition data
Order of preference: a cached array file from an earlier session, an already extracted folder, the zip on Drive or in `/content`, and finally an upload prompt. Decoded images are cached so later sessions skip the unzip.

In [ ]:
DATASET_CACHE = CACHE_DIR / "dataset_uint8.npz"


def find_labels_file(root, max_depth=3):
    root = Path(root)
    if not root.exists():
        return None
    for current, subdirs, files in os.walk(root):
        depth = len(Path(current).relative_to(root).parts)
        if "train_labels.csv" in files and Path(current, "train_images").exists():
            return Path(current)
        if depth >= max_depth:
            subdirs.clear()
    return None


def locate_extracted_dataset():
    candidates = [DATA_DIR_OVERRIDE, LOCAL_WORK_DIR / "data", "/kaggle/input"]
    for candidate in candidates:
        if candidate:
            found = find_labels_file(candidate)
            if found:
                return found
    return None


EXPECTED_ZIP_BYTES = 125_358_316


def is_complete_zip(path):
    if zipfile.is_zipfile(path):
        return True
    size_mb = path.stat().st_size / 1e6
    print(f"Skipping {path}: it is not a complete zip ({size_mb:.1f} MB of about {EXPECTED_ZIP_BYTES / 1e6:.0f} MB). "
          "The upload probably did not finish. Delete it and upload again, or put the zip in Google Drive.")
    return False


def locate_or_upload_zip():
    candidates = [LOCAL_WORK_DIR / DATA_ZIP_NAME, PROJECT_DIR / DATA_ZIP_NAME, Path("/content/drive/MyDrive") / DATA_ZIP_NAME]
    for candidate in candidates:
        if candidate.exists() and is_complete_zip(candidate):
            return candidate
    if IN_COLAB:
        from google.colab import files

        print(f"Could not find a complete {DATA_ZIP_NAME}. Please upload it now (about 125 MB) and wait for 100%.")
        uploaded = files.upload()
        if uploaded:
            uploaded_path = LOCAL_WORK_DIR / next(iter(uploaded))
            if is_complete_zip(uploaded_path):
                return uploaded_path
    raise FileNotFoundError(f"No complete {DATA_ZIP_NAME} found. Put it in {PROJECT_DIR} on Google Drive and run this cell again.")


def extract_zip(zip_path, destination):
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(destination)
    return find_labels_file(destination)


def read_png(path):
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"), dtype=np.uint8)


def read_png_folder(folder, filenames):
    with ThreadPoolExecutor(max_workers=16) as pool:
        return np.stack(list(pool.map(read_png, [folder / name for name in filenames])))


def build_dataset_arrays(dataset_dir):
    train_table = pd.read_csv(dataset_dir / "train_labels.csv").sort_values("id")
    test_table = pd.read_csv(dataset_dir / "test.csv").sort_values("id")
    print("Decoding PNG files (one time only) ...")
    return {
        "train_images": read_png_folder(dataset_dir / "train_images", train_table["filename"]),
        "train_labels": train_table["label"].to_numpy(dtype=np.int64),
        "test_images": read_png_folder(dataset_dir / "test_images", test_table["filename"]),
        "test_ids": test_table["id"].to_numpy(dtype=np.int64),
    }


def load_dataset():
    if DATASET_CACHE.exists():
        print(f"Loading cached arrays from {DATASET_CACHE}")
        with np.load(DATASET_CACHE) as cached:
            return {key: cached[key] for key in cached.files}
    dataset_dir = locate_extracted_dataset() or extract_zip(locate_or_upload_zip(), LOCAL_WORK_DIR / "data")
    if dataset_dir is None:
        raise FileNotFoundError("The zip did not contain train_labels.csv and train_images/.")
    arrays = build_dataset_arrays(dataset_dir)
    np.savez(DATASET_CACHE, **arrays)
    return arrays


start = time.time()
DATA = load_dataset()
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
print(f"Loaded in {time.time() - start:.0f}s")
print(f"Train images: {DATA['train_images'].shape}, test images: {DATA['test_images'].shape}")
print("Images per class:", np.bincount(DATA["train_labels"], minlength=10).tolist())
assert DATA["train_images"].shape[1:] == (32, 32, 3) and DATA["test_images"].shape[1:] == (32, 32, 3)

## 6. Validation split (shared with the WRN run)
A stratified 45,000 / 5,000 split with a fixed seed. The indices are saved to the project folder; give `split_seed42_val_indices.npy` to the WRN teammate so both models are scored on the same 5,000 images. If that file already exists (for example, the teammate's copy), it is reused.

In [ ]:
def stratified_split(labels, val_per_class, seed):
    rng = np.random.default_rng(seed)
    val_indices = []
    for class_id in np.unique(labels):
        class_indices = np.flatnonzero(labels == class_id)
        val_indices.extend(rng.choice(class_indices, size=val_per_class, replace=False))
    val_indices = np.sort(np.asarray(val_indices))
    train_indices = np.setdiff1d(np.arange(len(labels)), val_indices)
    return train_indices, val_indices


split_file = PROJECT_DIR / f"split_seed{CONFIG['split_seed']}_val_indices.npy"
if split_file.exists():
    VAL_INDICES = np.load(split_file)
    TRAIN_INDICES = np.setdiff1d(np.arange(len(DATA["train_labels"])), VAL_INDICES)
    print(f"Reusing split from {split_file}")
else:
    TRAIN_INDICES, VAL_INDICES = stratified_split(DATA["train_labels"], CONFIG["val_per_class"], CONFIG["split_seed"])
    np.save(split_file, VAL_INDICES)
    print(f"Saved split to {split_file}")

if CONFIG["train_on_full"]:
    TRAIN_INDICES = np.arange(len(DATA["train_labels"]))
    print("Training on all 50,000 images. Validation numbers are skipped for this run because val is inside train.")

subset_rng = np.random.default_rng(SEED)
if CONFIG["train_subset"]:
    TRAIN_INDICES = np.sort(subset_rng.choice(TRAIN_INDICES, size=CONFIG["train_subset"], replace=False))
if CONFIG["val_subset"]:
    VAL_INDICES = np.sort(subset_rng.choice(VAL_INDICES, size=CONFIG["val_subset"], replace=False))

HAS_VALIDATION = not CONFIG["train_on_full"]
print(f"Train: {len(TRAIN_INDICES):,} images | Validation: {len(VAL_INDICES):,} images")

## 7. Augmentation and data loaders
`basic` = reflect-pad crop + horizontal flip. `full` = basic + TrivialAugment + RandomGrayscale(0.05) + Cutout 16x16, which targets the brightness, contrast, sharpness and grayscale differences measured in the test set.

The **shifted validation copy** applies the measured test-set shift (brightness x1.04, contrast x1.03, 4% grayscale) to the validation images. It is a photometric sanity check only; clean validation stays the main metric.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
SHIFT_BRIGHTNESS, SHIFT_CONTRAST, SHIFT_GRAYSCALE_FRACTION = 1.04, 1.03, 0.04


class Cutout:
    def __init__(self, size):
        self.size = size

    def __call__(self, image):
        _, height, width = image.shape
        center_y, center_x = random.randrange(height), random.randrange(width)
        top, bottom = max(0, center_y - self.size // 2), min(height, center_y + self.size // 2)
        left, right = max(0, center_x - self.size // 2), min(width, center_x + self.size // 2)
        image[:, top:bottom, left:right] = 0.0
        return image


class ArrayImageDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        label = -1 if self.labels is None else int(self.labels[index])
        return self.transform(Image.fromarray(self.images[index])), label


def build_train_transform(augmentation, cutout_size, grayscale_p):
    steps = [T.RandomCrop(32, padding=4, padding_mode="reflect"), T.RandomHorizontalFlip()]
    if augmentation == "full":
        steps += [T.TrivialAugmentWide(), T.RandomGrayscale(p=grayscale_p)]
    steps += [T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)]
    if augmentation == "full":
        steps.append(Cutout(cutout_size))
    return T.Compose(steps)


def make_shifted_copy(images, brightness, contrast, grayscale_fraction, seed):
    rng = np.random.default_rng(seed)
    grayscale_indices = set(rng.choice(len(images), size=int(round(grayscale_fraction * len(images))), replace=False))
    shifted = np.empty_like(images)
    for index, array in enumerate(images):
        image = Image.fromarray(array)
        image = ImageEnhance.Brightness(image).enhance(brightness)
        image = ImageEnhance.Contrast(image).enhance(contrast)
        if index in grayscale_indices:
            image = image.convert("L").convert("RGB")
        shifted[index] = np.asarray(image)
    return shifted


def to_normalized_tensor(images):
    tensor = torch.from_numpy(images).permute(0, 3, 1, 2).float().div(255.0)
    mean = torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(1, 3, 1, 1)
    return ((tensor - mean) / std).contiguous(memory_format=torch.channels_last).to(DEVICE)


TRAIN_TRANSFORM = build_train_transform(CONFIG["augmentation"], CONFIG["cutout_size"], CONFIG["grayscale_p"])
train_dataset = ArrayImageDataset(DATA["train_images"][TRAIN_INDICES], DATA["train_labels"][TRAIN_INDICES], TRAIN_TRANSFORM)
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=True,
    prefetch_factor=4,
)

val_images = DATA["train_images"][VAL_INDICES]
VAL_LABELS = DATA["train_labels"][VAL_INDICES]
VAL_INPUTS = to_normalized_tensor(val_images)
SHIFTED_VAL_INPUTS = to_normalized_tensor(
    make_shifted_copy(val_images, SHIFT_BRIGHTNESS, SHIFT_CONTRAST, SHIFT_GRAYSCALE_FRACTION, CONFIG["split_seed"])
)
TEST_INPUTS = to_normalized_tensor(DATA["test_images"])

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Evaluation tensors on {DEVICE}: val {tuple(VAL_INPUTS.shape)}, test {tuple(TEST_INPUTS.shape)}")

In [ ]:
def show_augmented_samples(dataset, count=16):
    figure, axes = plt.subplots(2, count // 2, figsize=(count * 0.9, 2.2))
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)
    for axis, index in zip(axes.flat, np.random.default_rng(0).choice(len(dataset), count, replace=False)):
        image, label = dataset[index]
        axis.imshow((image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
        axis.set_title(CLASS_NAMES[label], fontsize=7)
        axis.axis("off")
    figure.suptitle(f"Training augmentation: {CONFIG['augmentation']}", fontsize=9)
    plt.tight_layout()
    plt.show()


show_augmented_samples(train_dataset)

## 8. Model: Trimmed ResNet-18 (submitted) and WideResNet teacher (never submitted)
The handout's residual block, `ReLU(S(x) + F(x))` with `F = conv3x3-BN-ReLU-conv3x3-BN`, stacked [2, 2, 2, 2]. Channels 64 / 128 / 256 / 224: ResNet-18's first three stages are kept at full width and only the parameter-heavy fourth stage is trimmed. The last BN in each residual branch starts at zero so every block begins as an identity map.

The teacher is a standard pre-activation WideResNet-28-10 (Zagoruyko and Komodakis, 2016), trained from scratch on the same 45,000 images. It is far over 5M parameters, so it is only used to produce soft targets during distillation; the parameter check below only applies to models that get submitted.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        residual = F.relu(self.bn1(self.conv1(x)))
        residual = self.bn2(self.conv2(residual))
        return F.relu(residual + self.shortcut(x))


class TrimmedResNet18(nn.Module):
    def __init__(self, stage_channels=(64, 128, 256, 224), blocks_per_stage=(2, 2, 2, 2), num_classes=10):
        super().__init__()
        stem_channels = stage_channels[0]
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_channels, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),
        )
        stages, in_channels = [], stem_channels
        for stage_index, (out_channels, num_blocks) in enumerate(zip(stage_channels, blocks_per_stage)):
            first_stride = 1 if stage_index == 0 else 2
            blocks = []
            for block_index in range(num_blocks):
                blocks.append(BasicBlock(in_channels, out_channels, first_stride if block_index == 0 else 1))
                in_channels = out_channels
            stages.append(nn.Sequential(*blocks))
        self.stages = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(in_channels, num_classes)
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
        for module in self.modules():
            if isinstance(module, BasicBlock):
                nn.init.zeros_(module.bn2.weight)

    def forward(self, x):
        features = self.stages(self.stem(x))
        return self.classifier(torch.flatten(self.pool(features), 1))


class WideBasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.shortcut = None
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False)

    def forward(self, x):
        activated = F.relu(self.bn1(x))
        residual = self.conv2(F.relu(self.bn2(self.conv1(activated))))
        skip = x if self.shortcut is None else self.shortcut(activated)
        return residual + skip


class WideResNet(nn.Module):
    def __init__(self, depth, widen_factor, num_classes=10):
        super().__init__()
        blocks_per_group = (depth - 4) // 6
        widths = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        self.stem = nn.Conv2d(3, widths[0], 3, stride=1, padding=1, bias=False)
        groups, in_channels = [], widths[0]
        for group_index, out_channels in enumerate(widths[1:]):
            first_stride = 1 if group_index == 0 else 2
            for block_index in range(blocks_per_group):
                groups.append(WideBasicBlock(in_channels, out_channels, first_stride if block_index == 0 else 1))
                in_channels = out_channels
        self.groups = nn.Sequential(*groups)
        self.final_bn = nn.BatchNorm2d(in_channels)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(in_channels, num_classes)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.Linear):
                nn.init.zeros_(module.bias)

    def forward(self, x):
        features = F.relu(self.final_bn(self.groups(self.stem(x))))
        return self.classifier(torch.flatten(self.pool(features), 1))


def build_model(config):
    architecture = config.get("architecture", "resnet18t")
    if architecture.startswith("wrn"):
        depth, widen_factor = (int(part) for part in architecture[3:].split("_"))
        return WideResNet(depth, widen_factor)
    return TrimmedResNet18(tuple(config["stage_channels"]), tuple(config["blocks_per_stage"]))


def count_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


PARAMETER_LIMIT = 5_000_000
summary_model = build_model(CONFIG).to(DEVICE)
summary(summary_model, (3, 32, 32), device=DEVICE.type)
NUM_PARAMETERS = count_trainable_parameters(summary_model)
del summary_model
IS_SUBMISSION_MODEL = CONFIG.get("role", "student") != "teacher"
print(f"\nTotal trainable parameters: {NUM_PARAMETERS:,} (limit {PARAMETER_LIMIT:,})")
if IS_SUBMISSION_MODEL:
    assert NUM_PARAMETERS <= PARAMETER_LIMIT, "Model is over the 5M parameter limit"
else:
    print("This is a TEACHER run: it is only used for distillation and will NOT produce a Kaggle submission.")

## 9. Training components
SGD with Nesterov momentum, or SAM (two passes per step; the second pass does not update BatchNorm statistics). Weight decay skips BatchNorm and bias parameters. Linear warmup then cosine decay, stepped every batch. An exponential moving average (EMA) of the weights is what we evaluate and submit. For distillation, the teacher is the average logit of several frozen models that see exactly the same augmented batch as the student.

In [ ]:
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer_class, rho=0.05, **base_kwargs):
        if rho <= 0:
            raise ValueError("rho must be positive")
        super().__init__(params, dict(rho=rho, **base_kwargs))
        self.base_optimizer = base_optimizer_class(self.param_groups, **base_kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def perturb_weights(self):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue
                perturbation = parameter.grad * scale
                parameter.add_(perturbation)
                self.state[parameter]["perturbation"] = perturbation

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for parameter in group["params"]:
                perturbation = self.state[parameter].pop("perturbation", None)
                if perturbation is not None:
                    parameter.sub_(perturbation)
        self.base_optimizer.step()

    def _grad_norm(self):
        norms = [parameter.grad.norm(p=2) for group in self.param_groups for parameter in group["params"] if parameter.grad is not None]
        return torch.norm(torch.stack(norms), p=2)

    def state_dict(self):
        return self.base_optimizer.state_dict()

    def load_state_dict(self, state_dict):
        self.base_optimizer.load_state_dict(state_dict)
        self.param_groups = self.base_optimizer.param_groups


class ModelEma:
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for parameter in self.module.parameters():
            parameter.requires_grad_(False)
        self.decay = decay
        self.num_updates = 0

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        decay = min(self.decay, (1 + self.num_updates) / (10 + self.num_updates))
        for ema_value, model_value in zip(self.module.state_dict().values(), model.state_dict().values()):
            if ema_value.dtype.is_floating_point:
                ema_value.lerp_(model_value.detach(), 1.0 - decay)
            else:
                ema_value.copy_(model_value)


def build_parameter_groups(model, weight_decay):
    decay, no_decay = [], []
    for parameter in model.parameters():
        if parameter.requires_grad:
            (no_decay if parameter.ndim <= 1 else decay).append(parameter)
    return [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]


def build_optimizer(model, config):
    parameter_groups = build_parameter_groups(model, config["weight_decay"])
    sgd_settings = dict(lr=config["peak_lr"], momentum=config["momentum"], nesterov=True)
    if config["optimizer"] == "sam":
        return SAM(parameter_groups, torch.optim.SGD, rho=config["sam_rho"], **sgd_settings)
    return torch.optim.SGD(parameter_groups, **sgd_settings)


def build_scheduler(optimizer, total_steps, warmup_steps):
    def lr_factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)


def set_batchnorm_stat_updates(model, enabled):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            if enabled and hasattr(module, "saved_momentum"):
                module.momentum = module.saved_momentum
            elif not enabled:
                module.saved_momentum = module.momentum
                module.momentum = 0.0


class InputRenormalizer(nn.Module):
    def __init__(self, target_mean, target_std):
        super().__init__()
        source_mean, source_std = torch.tensor(CIFAR_MEAN), torch.tensor(CIFAR_STD)
        target_mean, target_std = torch.tensor(target_mean), torch.tensor(target_std)
        self.register_buffer("scale", (source_std / target_std).view(1, 3, 1, 1))
        self.register_buffer("shift", ((source_mean - target_mean) / target_std).view(1, 3, 1, 1))

    def forward(self, x):
        return x * self.scale + self.shift


class TeacherEnsemble(nn.Module):
    def __init__(self, teachers, renormalizers):
        super().__init__()
        self.teachers = nn.ModuleList(teachers)
        self.renormalizers = nn.ModuleList(renormalizers)

    def forward(self, x):
        logits = [teacher(renorm(x)).float() for teacher, renorm in zip(self.teachers, self.renormalizers)]
        return torch.stack(logits).mean(dim=0)


def resolve_project_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_DIR / path


def load_teacher(spec):
    path = resolve_project_path(spec["path"])
    if spec["kind"] == "ours":
        checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
        teacher = build_model(checkpoint["config"])
        teacher.load_state_dict(checkpoint["ema_state"])
    elif spec["kind"] == "torchscript":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            teacher = torch.jit.load(str(path), map_location=DEVICE)
    else:
        raise ValueError(f"Unknown teacher kind: {spec['kind']}")
    teacher = teacher.to(DEVICE).eval()
    for parameter in teacher.parameters():
        parameter.requires_grad_(False)
    renormalizer = nn.Identity()
    if "mean" in spec or "std" in spec:
        renormalizer = InputRenormalizer(spec.get("mean", CIFAR_MEAN), spec.get("std", CIFAR_STD))
    return teacher, renormalizer.to(DEVICE)


def build_teacher_ensemble(teacher_specs):
    missing = [spec["path"] for spec in teacher_specs if not resolve_project_path(spec["path"]).exists()]
    if missing:
        raise FileNotFoundError(
            "Distillation needs these teacher files in the project folder:\n  " + "\n  ".join(missing)
            + "\nTrain the E2 seeds first, and ask the WRN teammate for a TorchScript export (see the last section)."
        )
    loaded = [load_teacher(spec) for spec in teacher_specs]
    return TeacherEnsemble([teacher for teacher, _ in loaded], [renorm for _, renorm in loaded]).eval()


def classification_loss(student_logits, targets, teacher_logits=None):
    if teacher_logits is None:
        return F.cross_entropy(student_logits, targets)
    temperature = KD_TEMPERATURE
    soft_targets = F.softmax(teacher_logits / temperature, dim=1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    distillation = F.kl_div(student_log_probs, soft_targets, reduction="batchmean") * temperature ** 2
    return KD_ALPHA * distillation + (1.0 - KD_ALPHA) * F.cross_entropy(student_logits, targets)

## 10. Train and evaluate functions

In [ ]:
def autocast(enabled):
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=enabled)


def to_device(images, targets):
    images = images.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
    return images, targets.to(DEVICE, non_blocking=True)


def train_one_epoch(train_model, base_model, loader, optimizer, scheduler, scaler, ema, teacher, amp_enabled):
    train_model.train()
    uses_sam = isinstance(optimizer, SAM)
    loss_sum = torch.zeros((), device=DEVICE)
    correct = torch.zeros((), device=DEVICE)
    seen = 0
    for images, targets in loader:
        images, targets = to_device(images, targets)
        teacher_logits = None
        if teacher is not None:
            with torch.no_grad(), autocast(amp_enabled):
                teacher_logits = teacher(images).float()
        with autocast(amp_enabled):
            logits = train_model(images)
        loss = classification_loss(logits.float(), targets, teacher_logits)
        optimizer.zero_grad(set_to_none=True)
        if uses_sam:
            loss.backward()
            optimizer.perturb_weights()
            optimizer.zero_grad(set_to_none=True)
            set_batchnorm_stat_updates(base_model, enabled=False)
            with autocast(amp_enabled):
                perturbed_logits = train_model(images)
            classification_loss(perturbed_logits.float(), targets, teacher_logits).backward()
            set_batchnorm_stat_updates(base_model, enabled=True)
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        ema.update(base_model)
        loss_sum += loss.detach() * targets.size(0)
        correct += (logits.argmax(dim=1) == targets).sum()
        seen += targets.size(0)
    return loss_sum.item() / seen, 100.0 * correct.item() / seen


def mirror_logits(model, images):
    return (model(images).float() + model(torch.flip(images, dims=[3])).float()) / 2


def mirror_translate_logits(model, images):
    padded = F.pad(images, (1, 1, 1, 1), mode="reflect")
    shifted_views = [padded[:, :, 0:32, 0:32], padded[:, :, 2:34, 2:34]]
    translated = torch.stack([mirror_logits(model, view) for view in shifted_views]).mean(dim=0)
    return 0.5 * mirror_logits(model, images) + 0.5 * translated


@torch.no_grad()
def predict_logits(model, inputs, tta="none", batch_size=1000):
    model.eval()
    outputs = []
    for images in torch.split(inputs, batch_size):
        with autocast(USE_AMP):
            if tta == "multi":
                logits = mirror_translate_logits(model, images)
            elif tta == "flip":
                logits = mirror_logits(model, images)
            else:
                logits = model(images).float()
        outputs.append(logits.float().cpu())
    return torch.cat(outputs)


def accuracy(logits, labels):
    return 100.0 * float((logits.argmax(dim=1).numpy() == labels).mean())


def save_checkpoint(path, epoch, base_model, ema, optimizer, scheduler, scaler, history):
    temporary_path = path.with_suffix(".tmp")
    torch.save(
        {
            "epoch": epoch,
            "config": CONFIG,
            "model_state": base_model.state_dict(),
            "ema_state": ema.module.state_dict(),
            "ema_updates": ema.num_updates,
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "history": history,
        },
        temporary_path,
    )
    os.replace(temporary_path, path)


RESUME_KEYS = ("epochs", "augmentation", "optimizer", "train_on_full", "distill", "batch_size", "stage_channels")


def restore_checkpoint(path, base_model, ema, optimizer, scheduler, scaler):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    mismatched = [key for key in RESUME_KEYS if checkpoint["config"].get(key) != CONFIG.get(key)]
    if mismatched:
        raise RuntimeError(f"Checkpoint at {path} was made with different settings ({mismatched}). Set RESUME = False or change SEED.")
    base_model.load_state_dict(checkpoint["model_state"])
    ema.module.load_state_dict(checkpoint["ema_state"])
    ema.num_updates = checkpoint["ema_updates"]
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    if scaler.is_enabled() and checkpoint["scaler_state"]:
        scaler.load_state_dict(checkpoint["scaler_state"])
    return checkpoint["epoch"] + 1, checkpoint["history"]

## 11. Train
Prints one line per epoch. `val` is the live model and `val-ema` the averaged weights (the one we submit). If Colab disconnects, just Run all again: training resumes from the last checkpoint.

In [ ]:
seed_everything(SEED)
base_model = build_model(CONFIG).to(DEVICE).to(memory_format=torch.channels_last)
train_model = torch.compile(base_model) if USE_COMPILE else base_model
ema = ModelEma(base_model, CONFIG["ema_decay"])
optimizer = build_optimizer(base_model, CONFIG)
steps_per_epoch = len(train_loader)
scheduler = build_scheduler(optimizer, CONFIG["epochs"] * steps_per_epoch, CONFIG["warmup_epochs"] * steps_per_epoch)
amp_enabled = USE_AMP and not (CONFIG["optimizer"] == "sam" and AMP_DTYPE == torch.float16)
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled and AMP_DTYPE == torch.float16)

teacher = None
if CONFIG["distill"]:
    teacher = build_teacher_ensemble(TEACHERS)
    for spec, single_teacher, renorm in zip(TEACHERS, teacher.teachers, teacher.renormalizers):
        single_accuracy = accuracy(predict_logits(nn.Sequential(renorm, single_teacher), VAL_INPUTS), VAL_LABELS)
        print(f"Teacher {spec['name']}: val accuracy {single_accuracy:.2f}%")
        if single_accuracy < 80:
            print("  Low accuracy: check this teacher's input normalization ('mean'/'std' in its spec).")
    print(f"Teacher ensemble: val accuracy {accuracy(predict_logits(teacher, VAL_INPUTS), VAL_LABELS):.2f}%")
    print("(If a teacher was trained on all 50k images, its val number is inflated.)")

last_checkpoint = RUN_DIR / "last.pth"
start_epoch, history = 0, []
if RESUME and last_checkpoint.exists():
    start_epoch, history = restore_checkpoint(last_checkpoint, base_model, ema, optimizer, scheduler, scaler)
    print(f"Resumed from epoch {start_epoch} of {CONFIG['epochs']}")

best_val_ema = max([row["val_acc_ema"] for row in history if row["val_acc_ema"] is not None], default=0.0)
run_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):
    epoch_start = time.time()
    train_loss, train_acc = train_one_epoch(train_model, base_model, train_loader, optimizer, scheduler, scaler, ema, teacher, amp_enabled)
    val_acc = val_acc_ema = None
    if HAS_VALIDATION:
        val_acc = accuracy(predict_logits(base_model, VAL_INPUTS), VAL_LABELS)
        val_acc_ema = accuracy(predict_logits(ema.module, VAL_INPUTS), VAL_LABELS)
    epoch_seconds = time.time() - epoch_start
    history.append({
        "epoch": epoch + 1,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "val_acc_ema": val_acc_ema,
        "epoch_seconds": epoch_seconds,
    })
    val_text = f"val {val_acc:5.2f}% | val-ema {val_acc_ema:5.2f}%" if HAS_VALIDATION else "val n/a (full-data run)"
    print(f"epoch {epoch + 1:3d}/{CONFIG['epochs']} | lr {history[-1]['lr']:.4f} | loss {train_loss:.4f} | "
          f"train {train_acc:5.2f}% | {val_text} | {epoch_seconds:4.1f}s")
    if epoch == start_epoch:
        remaining_minutes = epoch_seconds * (CONFIG["epochs"] - epoch - 1) / 60
        print(f"  Estimated time left: about {remaining_minutes:.0f} minutes")
    if HAS_VALIDATION and val_acc_ema > best_val_ema:
        best_val_ema = val_acc_ema
        torch.save({"config": CONFIG, "ema_state": ema.module.state_dict(), "epoch": epoch + 1, "val_acc_ema": val_acc_ema}, RUN_DIR / "best.pth")
    is_last_epoch = epoch + 1 == CONFIG["epochs"]
    if is_last_epoch or (epoch + 1) % CHECKPOINT_EVERY_EPOCHS == 0:
        save_checkpoint(last_checkpoint, epoch, base_model, ema, optimizer, scheduler, scaler, history)

with open(RUN_DIR / "history.json", "w") as handle:
    json.dump(history, handle, indent=1)
print(f"Training finished. This session trained for {(time.time() - run_start) / 60:.1f} minutes.")

## 12. Training curves

In [ ]:
def plot_history(history, save_path):
    frame = pd.DataFrame(history)
    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(frame["epoch"], frame["train_loss"])
    axes[0].set(title="Training loss", xlabel="epoch")
    axes[1].plot(frame["epoch"], frame["train_acc"], label="train (augmented)")
    if frame["val_acc"].notna().any():
        axes[1].plot(frame["epoch"], frame["val_acc"], label="val (live weights)")
        axes[1].plot(frame["epoch"], frame["val_acc_ema"], label="val (EMA weights)")
        best_row = frame.loc[frame["val_acc_ema"].idxmax()]
        late_phase = frame.iloc[len(frame) // 10:]
        axis_floor = min(late_phase["train_acc"].min(), late_phase["val_acc"].min(), late_phase["val_acc_ema"].min())
        axes[1].set_ylim(max(0, axis_floor - 2), 100)
        axes[1].set_title(f"Accuracy (best val EMA {best_row['val_acc_ema']:.2f}% @ epoch {int(best_row['epoch'])})")
    else:
        axes[1].set_title("Accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    axes[2].plot(frame["epoch"], frame["lr"])
    axes[2].set(title="Learning rate (end of epoch)", xlabel="epoch")
    figure.suptitle(RUN_NAME)
    plt.tight_layout()
    figure.savefig(save_path, dpi=150)
    plt.show()


plot_history(history, RUN_DIR / "curves.png")

## 13. Final validation results
The submitted model is the **final EMA checkpoint** (fixed in advance, not the best epoch on validation, so the validation number stays honest). Two test-time augmentation options are scored on validation: `flip` (image + mirror) and `multi` (mirror plus two 1-pixel shifts, as in airbench by Keller Jordan). The better one on validation is used for the shifted check and the Kaggle file.

In [ ]:
final_model = ema.module
results = {
    "run": RUN_NAME,
    "experiment": EXPERIMENT,
    "seed": SEED,
    "optimizer": CONFIG["optimizer"],
    "augmentation": CONFIG["augmentation"],
    "epochs": CONFIG["epochs"],
    "distill": CONFIG["distill"],
    "train_on_full": CONFIG["train_on_full"],
    "parameters": NUM_PARAMETERS,
    "mean_epoch_seconds": float(np.mean([row["epoch_seconds"] for row in history])),
}

TTA_MODE = "multi"
if HAS_VALIDATION:
    flip_val_logits = predict_logits(final_model, VAL_INPUTS, tta="flip")
    multi_val_logits = predict_logits(final_model, VAL_INPUTS, tta="multi")
    results["val_acc_ema_flip"] = accuracy(flip_val_logits, VAL_LABELS)
    results["val_acc_ema_multi"] = accuracy(multi_val_logits, VAL_LABELS)
    TTA_MODE = "multi" if results["val_acc_ema_multi"] >= results["val_acc_ema_flip"] else "flip"
    val_logits = multi_val_logits if TTA_MODE == "multi" else flip_val_logits
    shifted_logits = predict_logits(final_model, SHIFTED_VAL_INPUTS, tta=TTA_MODE)
    results["tta_mode"] = TTA_MODE
    results["val_acc_live"] = accuracy(predict_logits(base_model, VAL_INPUTS), VAL_LABELS)
    results["val_acc_ema"] = accuracy(predict_logits(final_model, VAL_INPUTS), VAL_LABELS)
    results["val_acc_ema_tta"] = accuracy(val_logits, VAL_LABELS)
    results["shifted_val_acc_ema_tta"] = accuracy(shifted_logits, VAL_LABELS)
    results["projected_leaderboard_if_cifar10_1_like"] = 1.62 * results["val_acc_ema_tta"] - 65.51
    np.save(RUN_DIR / "val_logits.npy", val_logits.numpy())

    predictions = val_logits.argmax(dim=1).numpy()
    per_class = pd.DataFrame({
        "class": CLASS_NAMES,
        "val accuracy %": [100.0 * (predictions[VAL_LABELS == c] == c).mean() for c in range(10)],
    })
    print(per_class.round(2).to_string(index=False))

print("\n" + "=" * 60)
print(f"Run:                          {RUN_NAME}")
print(f"Trainable parameters:         {NUM_PARAMETERS:,}")
if HAS_VALIDATION:
    print(f"Validation accuracy (live):   {results['val_acc_live']:.2f}%")
    print(f"Validation accuracy (EMA):    {results['val_acc_ema']:.2f}%")
    print(f"  with flip TTA:              {results['val_acc_ema_flip']:.2f}%")
    print(f"  with multi-crop TTA:        {results['val_acc_ema_multi']:.2f}%  -> using '{TTA_MODE}'")
    print(f"FINAL validation acc (EMA+TTA): {results['val_acc_ema_tta']:.2f}%")
    print(f"Shifted-val acc (EMA+TTA):    {results['shifted_val_acc_ema_tta']:.2f}% "
          f"(drop {results['val_acc_ema_tta'] - results['shifted_val_acc_ema_tta']:.2f} pts)")
    if results["val_acc_ema_tta"] >= 85:
        print(f"Projected leaderboard*:       {results['projected_leaderboard_if_cifar10_1_like']:.1f}%")
        print("* only if the Kaggle test set behaves like CIFAR-10.1 (Recht et al. 2018 fit)")
    else:
        print("Projected leaderboard:        n/a (the fit is only meaningful for well-trained models)")
else:
    print("Validation: not available (this run trained on all 50,000 images)")
print("=" * 60)

with open(RUN_DIR / "results.json", "w") as handle:
    json.dump(results, handle, indent=1)
results_log = PROJECT_DIR / "results_log.csv"
previous_results = pd.read_csv(results_log) if results_log.exists() else pd.DataFrame()
if not previous_results.empty:
    previous_results = previous_results[previous_results["run"] != RUN_NAME]
pd.concat([previous_results, pd.DataFrame([results]).round(3)], ignore_index=True).to_csv(results_log, index=False)
print(f"Updated {results_log} (one row per run, compare runs there)")

## 14. Kaggle submission
Writes `submission.csv` in the exact `id,label` format of `sample_submission.csv`. Upload it on the competition page under *Submit Prediction*. Pick your two final Kaggle selections by validation accuracy, not by public leaderboard rank.

In [ ]:
test_logits = predict_logits(final_model, TEST_INPUTS, tta=TTA_MODE)
np.save(RUN_DIR / "test_logits.npy", test_logits.numpy())

if not IS_SUBMISSION_MODEL or NUM_PARAMETERS > PARAMETER_LIMIT:
    print("Teacher run: no submission file written. Teachers are only used for distillation; the WRN-28-10 is far over the 5M limit.")
    print(f"Next step: set EXPERIMENT = 'DISTILL_wrn' and Run all. The teacher checkpoint is {RUN_DIR / 'last.pth'}")
else:
    submission = pd.DataFrame({"id": DATA["test_ids"], "label": test_logits.argmax(dim=1).numpy()})
    assert len(submission) == len(DATA["test_images"]) and submission["id"].is_unique
    assert submission["label"].between(0, 9).all()

    submission_path = RUN_DIR / "submission.csv"
    submission.to_csv(submission_path, index=False)
    local_copy = LOCAL_WORK_DIR / f"submission_{RUN_NAME}_{TTA_MODE}tta.csv"
    submission.to_csv(local_copy, index=False)

    print(f"Saved {submission_path} (test-time augmentation: {TTA_MODE})")
    print("Predicted images per class:", np.bincount(submission["label"], minlength=10).tolist())
    print(submission.head())

    if IN_COLAB and AUTO_DOWNLOAD_SUBMISSION:
        from google.colab import files

        files.download(str(local_copy))

## 15. Export this model as a teacher (TorchScript)
Saves the final EMA model as `model_scripted.pt`, which loads without our class definitions. The WRN teammate should run the same few lines on their model and drop the file at `MyDrive/DL-ResNet/teachers/wrn34_3_scripted.pt` (or share it and you place it there):

```python
model.eval()
scripted = torch.jit.trace(model.cpu(), torch.randn(1, 3, 32, 32))
scripted.save("wrn34_3_scripted.pt")
```

If their model was trained with different normalization numbers, add them to the teacher entry in the config cell, for example `{"name": "wrn34_3", "kind": "torchscript", "path": "teachers/wrn34_3_scripted.pt", "mean": [0.4914, 0.4822, 0.4465], "std": [0.2023, 0.1994, 0.2010]}`.

In [ ]:
export_model = copy.deepcopy(final_model).float().cpu().eval().to(memory_format=torch.contiguous_format)
scripted_path = RUN_DIR / "model_scripted.pt"
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    scripted_model = torch.jit.trace(export_model, torch.randn(1, 3, 32, 32))
    scripted_model.save(str(scripted_path))
    reloaded = torch.jit.load(str(scripted_path))
check_input = torch.randn(4, 3, 32, 32)
assert torch.allclose(reloaded(check_input), export_model(check_input), atol=1e-4)
print(f"Saved TorchScript teacher to {scripted_path}")